> 在线讲义：[第 15 章 · iLQR 与 MPPI：不训练策略，每次现算一条轨迹](https://wikieden.github.io/robot-motion-rl/lessons/15-ilqr-mppi.html) · 本书《机器人运控与强化学习》

# 第 15 章 · iLQR 与 MPPI：不训练策略，每次现算一条轨迹

配套讲义：仓库里 `robot-motion-rl/notes/lessons/15-ilqr-mppi.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（系统和代价是编的；MPPI 随机种子固定为 0）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 不控制 | 总代价 | `total_cost(rollout(0, zeros), zeros)` | 296.09 |
| 第二步 iLQR | 反推 + 正推 | `ilqr()` | 296.09 → 39.04 → 23.76 → 23.73；末端 x = 3.141 |
| 第三步 权重 | exp(−ΔS/λ) | `mppi_weights([10, 12, 20, 11], λ)` | λ=1：[0.665, 0.09, 0, 0.245] |
| 第三步 MPPI | 8 轮 | `mppi()` | 296.1 → … → 25.0 |

例子（编的）：x_{t+1} = x_t + dt·(sin x_t + u_t)，从 x=0 走到 x=π（像把摆从下面甩到上面），代价 Σ q(x−π)² + r·u²。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
DT, Q, R, T = 0.1, 1.0, 0.1, 30

**函数 `f`**：非线性动力学。

In [ ]:
def f(x, u):
    """非线性动力学。"""
    return x + DT * (np.sin(x) + u)

**函数 `total_cost`**：轨迹总代价：Σ q(x−π)² + r·u²（x 取第 1 到 T 步）。

In [ ]:
def total_cost(xs, us):
    """轨迹总代价：Σ q(x−π)² + r·u²（x 取第 1 到 T 步）。"""
    return float(Q * np.sum((np.asarray(xs[1:]) - np.pi) ** 2) + R * np.sum(np.square(us)))

**函数 `rollout`**：按控制序列往前滚，返回状态序列。

In [ ]:
def rollout(x0, us):
    """按控制序列往前滚，返回状态序列。"""
    xs = [x0]
    for u in us:
        xs.append(f(xs[-1], u))
    return np.array(xs)

**函数 `ilqr`**：一维 iLQR：反推（线性化 a = ∂f/∂x，b = ∂f/∂u，标量 Riccati）→ 正推（带线搜索），返回每次迭代的代价。

In [ ]:
def ilqr(x0=0.0, iters=8):
    """一维 iLQR：反推（线性化 a = ∂f/∂x，b = ∂f/∂u，标量 Riccati）→ 正推（带线搜索），返回每次迭代的代价。"""
    us = np.zeros(T)
    xs = rollout(x0, us)
    costs = [total_cost(xs, us)]
    for _ in range(iters):
        vx, vxx = 0.0, 0.0
        ks, kk = np.zeros(T), np.zeros(T)
        for t in reversed(range(T)):
            a, b = 1 + DT * np.cos(xs[t]), DT
            lx = 2 * Q * (xs[t + 1] - np.pi)
            # 把 t+1 步的状态代价折进来：Q 函数对 (δx, δu) 的二次展开
            qx = a * (vx + lx)
            qu = 2 * R * us[t] + b * (vx + lx)
            qxx = a * a * (vxx + 2 * Q)
            quu = 2 * R + b * b * (vxx + 2 * Q)
            qux = b * a * (vxx + 2 * Q)
            ks[t], kk[t] = -qu / quu, -qux / quu
            vx = qx + kk[t] * quu * ks[t] + kk[t] * qu + qux * ks[t]
            vxx = qxx + kk[t] * quu * kk[t] + 2 * kk[t] * qux
        best = None
        for alpha in (1.0, 0.5, 0.25, 0.1):
            xn, un = [x0], np.zeros(T)
            for t in range(T):
                un[t] = us[t] + alpha * ks[t] + kk[t] * (xn[-1] - xs[t])
                xn.append(f(xn[-1], un[t]))
            c = total_cost(xn, un)
            if c < costs[-1]:
                best = (np.array(xn), un, c)
                break
        if best is None:
            break
        xs, us, c = best
        costs.append(c)
    return costs, xs

**函数 `mppi_weights`**：MPPI 权重：w_k ∝ exp(−(S_k − min S)/λ)，归一化到和为 1。

In [ ]:
def mppi_weights(costs, lam):
    """MPPI 权重：w_k ∝ exp(−(S_k − min S)/λ)，归一化到和为 1。"""
    s = np.asarray(costs, float)
    w = np.exp(-(s - s.min()) / lam)
    return w / w.sum()

**函数 `mppi`**：MPPI：每轮在当前控制序列上加噪声采样，按权重平均噪声更新序列，返回每轮代价。

In [ ]:
def mppi(x0=0.0, samples=64, sigma=1.0, lam=1.0, iters=8, seed=0):
    """MPPI：每轮在当前控制序列上加噪声采样，按权重平均噪声更新序列，返回每轮代价。"""
    rng = np.random.default_rng(seed)
    us = np.zeros(T)
    costs = [total_cost(rollout(x0, us), us)]
    for _ in range(iters):
        eps = rng.normal(0, sigma, (samples, T))
        s = [total_cost(rollout(x0, us + e), us + e) for e in eps]
        us = us + mppi_weights(s, lam) @ eps
        costs.append(total_cost(rollout(x0, us), us))
    return costs

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 iLQR")
costs, xs = ilqr()
print("每次迭代的代价：", [round(c, 2) for c in costs])
print(f"最后一步 x = {xs[-1]:.3f}（目标 π ≈ 3.142）")

In [ ]:
print("\n第三步 MPPI 的权重")
s = [10.0, 12.0, 20.0, 11.0]
for lam in (1.0, 5.0):
    print(f"代价 {s}，λ={lam} → 权重 {np.round(mppi_weights(s, lam), 3)}")

In [ ]:
print("\n第三步 MPPI 迭代")
print("每轮代价：", [round(c, 1) for c in mppi()])

## 3. 代码里才看得到的

- iLQR 的反推里，第 t 步的状态代价算在 $x_{t+1}$ 上，所以要通过 a、b 折回到 $(x_t, u_t)$；这是代码里 `lx`、`qx`、`qu` 那几行的来源。
- MPPI 的权重先减去最小代价再取 exp，防止 exp(−296) 下溢成 0；这不改变归一化后的权重。

## 4. 自己改一改

- 把 MPPI 的采样条数从 64 改成 8 或 512，看收敛速度和最终代价怎么变。
- 把 iLQR 的初始控制从全 0 改成全 1，看收敛到的代价是否相同。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 不练本能现规划，求一串控制量；
> iLQR 沿轨迹线性化，LQR 往回推，三遍就收敛；
> MPPI 撒一把，exp 负代价除 λ 加权平均，不要导数好并行；
> 在线现算是 MPC，离线学好是 RL。